In [1]:
import camelot
import pandas as pd
import bs4 as BeautifulSoup
import requests
import json

c:\Users\Tan_S\AppData\Local\Programs\Python\Python311\Lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [2]:
overall_data = {
    'Station Name': [],
    'Latitude': [],
    'Longitude': [],
    'Station ID': []
}
print(overall_data)
all_station_data = pd.DataFrame(overall_data)

{'Station Name': [], 'Latitude': [], 'Longitude': [], 'Station ID': []}


In [3]:
## getting station id from data.gov.sg api
url_list = [
    "https://api-open.data.gov.sg/v2/real-time/api/air-temperature",
    "https://api-open.data.gov.sg/v2/real-time/api/rainfall",
    "https://api-open.data.gov.sg/v2/real-time/api/wind-direction",
    "https://api-open.data.gov.sg/v2/real-time/api/wind-speed",
    "https://api-open.data.gov.sg/v2/real-time/api/relative-humidity"]

responses = [requests.get(url) for url in url_list]
json_data = [response.json() for response in responses]

for i in range(len(url_list)):
    data = json_data[i]
    for station in data['data']['stations']:
        if station['id'] not in overall_data['Station ID']:
            name = station['name']
            id = station['id']
            lat = station['location']['latitude']
            long = station['location']['longitude']

            overall_data['Station Name'].append(name)
            overall_data['Latitude'].append(lat)
            overall_data['Longitude'].append(long)
            overall_data['Station ID'].append(id)
        else:
            continue

In [4]:
## sanity check: number of stations collected in overall_data
len(overall_data['Station Name'])

60

In [5]:
all_station_data = pd.DataFrame(overall_data)

In [6]:
all_station_data

,Station Name,Latitude,Longitude,Station ID
0,Ang Mo Kio Avenue 5,1.37640,103.84920,S109
1,Pulau Ubin,1.41680,103.96730,S106
2,Banyan Road,1.25600,103.67900,S117
3,East Coast Parkway,1.31350,103.96250,S107
4,Tuas South Avenue 3,1.29377,103.61843,S115
5,Semakau Landfill,1.18900,103.76800,S102
6,Sentosa,1.25000,103.82790,S60
7,Clementi Road,1.33370,103.77680,S50
8,Nanyang Avenue,1.34583,103.68166,S44
9,Kim Chuan Road,1.33990,103.88780,S43


In [7]:
########################################################################################
# do not change
lat_min, lat_max = 1.15, 1.47
lon_min, lon_max = 103.56, 104.14   
########################################################################################

########################################################################################
# do not change
scale_x = 217 / 853
scale_y = 120 / 479
########################################################################################


def latlon_to_xy(lat, lon, lat_min, lat_max, lon_min, lon_max, width, height):
    """
    Convert latitude and longitude to pixel coordinates on the radar image grid.

    Args:
        lat, lon : float
        lat_min, lat_max, lon_min, lon_max : float
            Bounding box of the radar coverage.
        width, height : int
            Dimensions of the radar image (e.g. 217x120).

    Returns:
        (x, y) pixel coordinates (integers)
    """

    # normalize longitude to horizontal position
    x = (lon - lon_min) / (lon_max - lon_min) * (width - 1)
    # normalize latitude to vertical position (note inversion: north is top)
    y = (lat_max - lat) / (lat_max - lat_min) * (height - 1)

    return int(round(x)), int(round(y))

def basemap_to_radar(X, Y):
    x = X * scale_x
    y = Y * scale_y
    return int(round(x)), int(round(y))

all_station_data['coord_basemap'] = all_station_data.apply(lambda row : latlon_to_xy(row['Latitude'],row['Longitude'],lat_min, lat_max, lon_min, lon_max,853,479),axis = 1)

all_station_data.rename(columns={'Station Name':'station_name','Latitude':'latitude','Longitude':'longitude','Station ID':'station_id'},inplace = True)


all_station_data["coord"] = all_station_data["coord_basemap"].apply(lambda xy: basemap_to_radar(xy[0], xy[1]))

In [8]:
all_station_data

,station_name,latitude,longitude,station_id,coord_basemap,coord
0,Ang Mo Kio Avenue 5,1.37640,103.84920,S109,"(425, 140)","(108, 35)"
1,Pulau Ubin,1.41680,103.96730,S106,"(598, 79)","(152, 20)"
2,Banyan Road,1.25600,103.67900,S117,"(175, 320)","(45, 80)"
3,East Coast Parkway,1.31350,103.96250,S107,"(591, 234)","(150, 59)"
4,Tuas South Avenue 3,1.29377,103.61843,S115,"(86, 263)","(22, 66)"
5,Semakau Landfill,1.18900,103.76800,S102,"(306, 420)","(78, 105)"
6,Sentosa,1.25000,103.82790,S60,"(394, 329)","(100, 82)"
7,Clementi Road,1.33370,103.77680,S50,"(318, 204)","(81, 51)"
8,Nanyang Avenue,1.34583,103.68166,S44,"(179, 185)","(46, 46)"
9,Kim Chuan Road,1.33990,103.88780,S43,"(482, 194)","(123, 49)"
